In [ ]:
area = "extremadura"
scenario = "C6" #par

In [ ]:
xlsx = f"/code/data/results/{area}/{scenario}/lri_model/models.xlsx"
sheet = 'models'

cweight = {1: 1, 0 : 0.25} # se a classe 0 estiver muito mais representada do que a classe 1
#cweight = None #atribuir os mesmo pesos

In [ ]:
import os
import datetime as dt
from glass.rst.cls import trainfeat_to_npy, npy_to_mdl
from glass.rd import tbl_to_obj

In [ ]:
df = tbl_to_obj(xlsx, sheet)

In [ ]:
df

In [ ]:
import os
import pandas as pd
import rasterio as rio

r = df.iloc[0]

featdf = tbl_to_obj(
    xlsx,
    r["name"]
)

files = {
    "trainref": r.trainref
}

for feat in featdf.trainfeat.tolist():
    files[feat] = os.path.join(
        r.trainfeat,
        feat
    )

info = []

for name, path in files.items():

    with rio.open(path) as src:
        info.append({
            "raster": name,
            "path": path,
            "shape": src.shape,
            "resolução": src.res,
            "crs": str(src.crs),
            "transform": src.transform
        })

info_df = pd.DataFrame(info)

info_df[
    [
        "raster",
        "shape",
        "resolução",
        "crs"
    ]
]

In [ ]:
for i, r in df.iterrows():
    if r.status != 'run':
        continue

    time_a = dt.datetime.now().replace(microsecond=0)
    featdf = tbl_to_obj(xlsx, r['name'])

    feats = featdf.trainfeat.tolist()
    paths = [os.path.join(r.trainfeat, f) for f in feats]

    trainfeat_to_npy(r.trainref, paths, r.yfile, r.xfile) # usa o raster de reino para selecionar as células validas e vai buscar os valores das variáveis nessas mesmas posições 
    time_b = dt.datetime.now().replace(microsecond=0)

    print(time_b - time_a)

In [ ]:
for i, r in df.iterrows():
    if r.status != 'run':
        continue

    trees= int(r.ntrees)
    msamples = None if r.max_samples == -1 else r.max_samples
    npy_to_mdl(
        r.yfile, r.xfile, r.model, 
        trees=trees, samples=msamples,
        regressor=None,
        cweight=cweight
    )

    print("Modelo criado:", r.model)